# Comparing Vault static and dynamic secrets for cloud IAM credential management

HashiCorp Vault supports two models for credential management. Static secrets are stored values that clients read and use as-is. Dynamic secrets are generated on demand by Vault, held for a configurable lifetime, and revoked automatically when the lease expires. This notebook compares the two models in the context of cloud IAM credentials, where the choice between them has direct security and operational consequences.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
import json
import sys

MODEL = {
    "static": {
        "lifecycle": "client-managed",
        "rotation": "manual",
        "revocation": "explicit only",
        "audit": "read events",
        "blast_radius": "full secret value"
    },
    "dynamic": {
        "lifecycle": "Vault-managed",
        "rotation": "automatic via TTL",
        "revocation": "lease-based, immediate",
        "audit": "issue and revoke events",
        "blast_radius": "single credential instance"
    }
}
print(json.dumps(MODEL, indent=2))

## Purpose

This notebook provides a structured comparison of Vault's static and dynamic secrets models as applied to cloud IAM credential management. Cloud IAM credentials (AWS IAM users/tokens, Azure service principal keys, GCP service account keys) are long-lived by default; Vault's dynamic secrets engine converts them into short-lived, on-demand credentials.

## When to Use Each Model

### Static secrets are appropriate when:

- The secret value is managed externally and Vault acts as a secure store, not an issuer.
- The secret rarely changes and does not require per-request rotation.
- The consumer needs the full secret value at all times (e.g., a third-party API key that Vault cannot regenerate).

### Dynamic secrets are appropriate when:

- Credentials need automatic rotation with a short TTL.
- Each consumer should receive unique, scoped credentials that are revoked on shutdown.
- Compliance requirements mandate minimizing credential lifetime and blast radius.
- The workload is ephemeral (CI/CD job, short-lived container) and does not need persistent credentials.

## Prerequisites

- A running Vault instance with the relevant secrets engines enabled (KV v2 for static storage; AWS/Azure/GCP engine for dynamic IAM credentials).
- Vault configured with a cloud provider backend: privileged credentials that Vault uses to assume roles or create service principals.
- A client environment with the Vault CLI or an SDK (for example, `hvac` for Python) configured with `VAULT_ADDR` and `VAULT_TOKEN`.
- For dynamic credentials: a role configuration defining credential type, TTL, and policies.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Configure a Vault client for subsequent cells
VAULT_ADDR = "http://127.0.0.1:8200"
VAULT_TOKEN = "root"

# Using hvac (HashiCorp Vault SDK for Python)
# import hvac
# client = hvac.Client(url=VAULT_ADDR, token=VAULT_TOKEN)
# assert client.is_authenticated(), 'Authentication failed'

print('Vault client configuration prepared.')

## Steps

### Step 1: Store a static secret in KV v2

Static IAM credentials (long-lived access keys) can be stored under a KV v2 path:

```bash
vault kv put secret/cloud-iam/static \
  access_key=AKIA... \
  secret_key=wJalr... \
  region=us-east-1
```

The client reads these values directly. The credentials never change unless manually overwritten.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Simulate reading a static secret from KV v2
def read_static_secret(client, path="secret/cloud-iam/static"):
    """Return static credentials stored in KV v2."""
    # In practice: client.secrets.kv.v2.read_secret_version(path)
    return {
        "path": path,
        "access_key": "AKIA...",
        "secret_key": "wJalr...",
        "region": "us-east-1",
        "rotation": "manual",
        "ttl": None,
    }

static_creds = read_static_secret(None)
print("Static secret read from KV v2:")
for k, v in static_creds.items():
    print(f"  {k}: {v}")

### Step 2: Enable the cloud secrets engine and define a dynamic role

The AWS secrets engine (or equivalent for Azure/GCP) generates short-lived IAM credentials on demand:

```bash
vault secrets enable -path=aws-iam aws

vault write aws-iam/roles/ci-role \
  credential_type=iam_user \
  policy_arns=arn:aws:iam::aws:policy/ReadOnlyAccess \
  default_sts_ttl=1h \
  max_sts_ttl=24h
```

This role tells Vault to create IAM users with read-only access and a default lifetime of one hour.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Simulate defining a dynamic IAM role configuration
def define_dynamic_role(name, credential_type, policy_arns, default_ttl, max_ttl):
    """Return the role configuration as Vault would store it."""
    return {
        "role": name,
        "credential_type": credential_type,
        "policy_arns": policy_arns if isinstance(policy_arns, list) else [policy_arns],
        "default_sts_ttl": default_ttl,
        "max_sts_ttl": max_ttl,
        "generated_on_demand": True,
    }

role = define_dynamic_role(
    name="ci-role",
    credential_type="iam_user",
    policy_arns="arn:aws:iam::aws:policy/ReadOnlyAccess",
    default_ttl="1h",
    max_ttl="24h",
)
print("Dynamic role configuration:")
print(json.dumps(role, indent=2))

### Step 3: Generate dynamic credentials and inspect the result

```bash
$ vault read aws-iam/creds/ci-role
Key                  Value
---                  -----
lease_id             aws-iam/creds/ci-role/...
lease_duration       1h
lease_renewable      true
access_key           ASIA...
secret_key           wJalr...
session_token        ...
```

Each read produces a unique credential pair. The lease is renewable up to `max_sts_ttl`.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Simulate generating dynamic IAM credentials
import uuid

def generate_dynamic_credentials(role, ttl="1h"):
    """Simulate Vault generating a dynamic IAM credential pair."""
    return {
        "access_key": f"ASIA{uuid.uuid4().hex[:16]}",
        "secret_key": f"wJalr{uuid.uuid4().hex[:24]}",
        "session_token": uuid.uuid4().hex,
        "lease_id": f"aws-iam/creds/{role}/{uuid.uuid4().hex[:12]}",
        "lease_duration": ttl,
        "lease_renewable": True,
        "credential_type": "iam_user",
    }

creds = generate_dynamic_credentials("ci-role", ttl="1h")
print("Dynamic credentials generated:")
print(json.dumps(creds, indent=2))

### Step 4: Compare lifecycle characteristics

The following cell summarizes the structural differences between the two models across key operational dimensions.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Comparison table — run all previous cells first
print(f"{'Dimension':<20} {'Static':<30} {'Dynamic':<30}")
print('-' * 80)
for dim in ["lifecycle", "rotation", "revocation", "audit", "blast_radius"]:
    s = MODEL["static"][dim]
    d = MODEL["dynamic"][dim]
    print(f"{dim:<20} {s:<30} {d:<30}")

### Step 5: Revoke a dynamic credential

Unlike static secrets, dynamic credentials can be revoked immediately, regardless of their remaining TTL:

```bash
vault lease revoke <lease_id>
```

After revocation, the credential is invalid and the underlying IAM user is deleted by Vault.

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Simulate revoking a dynamic credential lease
def revoke_credential(lease_id):
    """Simulate lease revocation."""
    return {
        "lease_id": lease_id,
        "status": "revoked",
        "credential_invalidated": True,
        "immediate": True,
    }

result = revoke_credential(creds["lease_id"])
print("Revocation result:")
print(json.dumps(result, indent=2))

## Verify

To confirm that the dynamic secrets workflow is functioning correctly:

1. Request credentials from the dynamic role and verify the TTL matches the configured `default_sts_ttl`.
2. Use the generated credentials against the cloud provider API to confirm they are valid and carry the expected permissions.
3. Attempt to use the credentials after the lease expires (or revoke the lease early) and confirm rejection.
4. Check the Vault audit log for both the issue and revoke events.
5. Confirm that revoking one credential does not affect credentials generated for a different lease.

## Common Errors

- **Requesting dynamic credentials with no role configured:** The secrets engine must have at least one role before credential generation is possible. An empty roles list returns an error.
- **Exceeding `max_sts_ttl`:** The `default_sts_ttl` cannot exceed `max_sts_ttl`. Setting both to the same value disables renewal.
- **Static secret treated as dynamic:** Storing long-lived IAM keys in KV and expecting rotation is a mismatch. KV v2 versioning preserves history but does not rotate the secret value.
- **Not revoking leases on shutdown:** If a workload exits without revoking its lease, the generated credential remains valid until TTL expiry. Use a shutdown hook or Vault Agent to handle revocation.

## References

- HashiCorp Vault documentation — Secrets Engines overview
- HashiCorp Vault documentation — AWS Secrets Engine
- HashiCorp Vault documentation — KV Secrets Engine v2
- HashiCorp Vault documentation — Lease management and revocation

In [ ]:
# last_verified: 2026-09-17 · HashiCorp Vault n/a
# Summary: when to choose each model
print("=" * 60)
print("Static secrets:")
print("  - Use when Vault is a secure store, not an issuer.")
print("  - Best for: config values, third-party API keys.")
print()
print("Dynamic secrets:")
print("  - Use when Vault generates and rotates credentials.")
print("  - Best for: cloud IAM, database credentials, ephemeral workloads.")
print()
print("For cloud IAM credential management, dynamic secrets are the")
print("recommended default — they minimize credential lifetime and blast radius.")
print("=" * 60)